# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mfaiqdev/MLinternship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Dataset loaded successfully!")
print(df.shape)

Dataset loaded successfully!
(9841378, 31)


In [3]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [4]:
print(
    df[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_sessions",
            "ga4_engaged_sessions",
        ]
    ].describe()
)

       gsc_impressions    gsc_clicks  gsc_avg_position  ga4_sessions  \
count     9.841378e+06  9.841378e+06      3.611061e+06  6.822637e+06   
mean      2.851812e+01  8.350782e-02      1.582665e+01  1.905140e-01   
std       1.559266e+02  7.814341e-01      1.985603e+01  1.968750e+00   
min       0.000000e+00  0.000000e+00      0.000000e+00  0.000000e+00   
25%       0.000000e+00  0.000000e+00      3.742120e+00  0.000000e+00   
50%       0.000000e+00  0.000000e+00      7.500000e+00  0.000000e+00   
75%       6.000000e+00  0.000000e+00      2.020000e+01  0.000000e+00   
max       4.008400e+04  2.740000e+02      4.980000e+02  7.920000e+02   

       ga4_engaged_sessions  
count          6.822637e+06  
mean           4.331316e-03  
std            7.664665e-02  
min            0.000000e+00  
25%            0.000000e+00  
50%            0.000000e+00  
75%            0.000000e+00  
max            2.100000e+01  


In [5]:
print(df["ga4_data_available"].value_counts(dropna=False))

ga4_data_available
False    6408671
None     3018741
True      413966
Name: count, dtype: int64


In [6]:
print(df["gsc_data_available"].value_counts(dropna=False))

gsc_data_available
False    6230317
True     3611061
Name: count, dtype: int64


In [6]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


This lane focuses on ranking content items that may require refresh attention.

I selected Random Forest Classifier because the problem is a binary decision:
whether a content item shows signals associated with refresh opportunity.

Random Forest was selected because it can capture non-linear relationships between SEO performance signals and provides feature importance for interpretation.

The model uses only historical performance features available at the decision moment.

The objective is not to prove causation, but to create a decision-support ranking that can be compared against the Week-4 rule baseline.

In [16]:
model_df["refresh_label"].value_counts()

,count
refresh_label,
0,9841378


In [17]:
y_train.value_counts()

,count
refresh_label,
0,8935676


In [19]:
import pandas as pd

baseline = pd.read_csv(
    "work/outputs/baseline_action_score.csv"
)

baseline.head()

FileNotFoundError: [Errno 2] No such file or directory: 'work/outputs/baseline_action_score.csv'

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

A grouped split is used where client_hash_id defines the groups.

This prevents the same client from appearing in both training and testing data.

The split reflects the real deployment scenario where the model should generalize to unseen content situations rather than memorize client-specific patterns.

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,refresh_label
0,20,0,3.350000,NaN,NaN,0
1,1,0,0.000000,NaN,NaN,0
2,125,1,4.928000,NaN,NaN,0
3,7,0,4.000000,NaN,NaN,0
4,11,0,2.272727,NaN,NaN,0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The model and baseline are evaluated on the same test split.

The metric used is precision because the business goal is reviewing a ranked queue.

High precision means fewer unnecessary content reviews.

In [10]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)


model.fit(
    X_train,
    y_train
)


model_predictions = model.predict(X_test)


print("Model trained")

Model trained


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


The model is evaluated beyond the final metric.

Error analysis focuses on:
- false positives: content incorrectly flagged for refresh
- false negatives: content missed by the model

Feature importance is reviewed to confirm the model relies on reasonable SEO signals.

The results are interpreted as decision-support signals rather than guaranteed recommendations.

,feature,importance
0,gsc_impressions,0.0
1,gsc_clicks,0.0
2,gsc_avg_position,0.0
3,ga4_sessions,0.0
4,ga4_engaged_sessions,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.